Code to analyze photometry data using a GLM. Input is a excel file or CSV. In our case, the first column was the dependent variable, and we defined x and y accordingly, but this can be adjusted.

Output will be first the R^2 of each model and then the R^2 of the model excluding each predictor one by one. Each drop-out will have a p-value associated with it, which runs a paired t-test on the predictions of the full vs reduced model. This will allow for easy assessment of whether dropping out a predictor deteriorates the model significantly.

The table at the bottom contains the Beta coefficients (coefficients for each predictor in the GLM equation) and p-values for each B coefficient. Note that this is for the full model without any drop-outs.


In [ ]:
import pandas as pd
import statsmodels.api as sm
import numpy as np
from google.colab import files
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.utils import resample
from scipy import stats

os.chdir('/content')

df = pd.read_excel('KLR.xlsx')

# drop NAN
df = df.replace([np.inf, -np.inf], np.nan).dropna()

# y is dependent vars, x independent vars
y = df.iloc[:, 0]  # First column as dependent variable (this may vary depending on your input)
X = df.iloc[:, 1:]  # Remaining columns as independent variables (this may vary depending on your input)

#train on 80%, test on 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# train GLM, get R^2 and the predictions
def train_glm_and_evaluate(X_train, y_train, X_test, y_test):
    X_train = sm.add_constant(X_train)
    X_test = sm.add_constant(X_test)

    glm_model = sm.GLM(y_train, X_train, family=sm.families.Gaussian())
    glm_results = glm_model.fit()

    y_pred = glm_results.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    return glm_results, r2, y_pred

# Function for bootstrapping R^2 confidence interval, standard deviation, and SEM
def bootstrap_r2(X, y, n_bootstraps=1000):
    r2_values = []
    for _ in range(n_bootstraps):
        # Resample data with replacement for bootstrapping
        X_resampled, y_resampled = resample(X, y, replace=True, random_state=np.random.randint(0, 10000))

        # Split into training and testing
        X_train_resampled, X_test_resampled, y_train_resampled, y_test_resampled = train_test_split(
            X_resampled, y_resampled, test_size=0.2, random_state=42
        )

        # Train model and compute R^2
        _, r2, _ = train_glm_and_evaluate(X_train_resampled, y_train_resampled, X_test_resampled, y_test_resampled)
        r2_values.append(r2)

    # Compute statistics
    mean_r2 = np.mean(r2_values)
    std_r2 = np.std(r2_values)
    sem_r2 = std_r2 / np.sqrt(n_bootstraps)
    lower_bound = np.percentile(r2_values, 2.5)
    upper_bound = np.percentile(r2_values, 97.5)

    return mean_r2, std_r2, sem_r2, lower_bound, upper_bound

# Train full model and get baseline accuracy
full_model_results, full_model_r2, full_model_preds = train_glm_and_evaluate(X_train, y_train, X_test, y_test)
full_model_mean_r2, full_model_std_r2, full_model_sem_r2, full_model_lower_ci, full_model_upper_ci = bootstrap_r2(X, y)

print(f"Full model R2: {full_model_mean_r2:.4f} (95% CI: [{full_model_lower_ci:.4f}, {full_model_upper_ci:.4f}]), STD: {full_model_std_r2:.4f}, SEM: {full_model_sem_r2:.4f}")

# Store the full model R2 and predictions
full_model_r2_values = [full_model_r2] * len(y_test)  # Store full model's R2
full_model_preds_values = full_model_preds  # Store full model's predictions

# Evaluate impact of dropping each independent variable
for column in X.columns:
    X_train_reduced = X_train.drop(columns=[column])
    X_test_reduced = X_test.drop(columns=[column])

    _, reduced_r2, reduced_preds = train_glm_and_evaluate(X_train_reduced, y_train, X_test_reduced, y_test)
    reduced_mean_r2, reduced_std_r2, reduced_sem_r2, reduced_lower_ci, reduced_upper_ci = bootstrap_r2(X_train_reduced, y_train)

    print(f"After dropping '{column}':")
    print(f"  R2: {reduced_mean_r2:.4f} (95% CI: [{reduced_lower_ci:.4f}, {reduced_upper_ci:.4f}]), STD: {reduced_std_r2:.4f}, SEM: {reduced_sem_r2:.4f}")

    # Paired t-test (p value for R2 full vs R2 reduced model)
    t_stat_r2, p_value_r2 = stats.ttest_rel(full_model_r2_values, [reduced_r2] * len(y_test))

    # Paired t-test for predictions comparison between full model and reduced model
    t_stat_preds, p_value_preds = stats.ttest_rel(full_model_preds_values, reduced_preds)

    #print(f"  p-value for R2 difference (full vs reduced): {p_value_r2:.4f}")
    print(f"  p-value (prediction difference for full vs dropout): {p_value_preds:.4f}")
    print()

# Print full model summary
print(full_model_results.summary())
